In [1]:
!nvidia-smi
!pip install -q transformers accelerate pillow datasets pandas tqdm pyarrow

Sun May 17 15:52:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import gc
import re
import json
import shutil
import torch
import pandas as pd

from pathlib import Path
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm
from PIL import Image
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor

In [3]:
# ============================================================
# 1. Config
# ============================================================

MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
DATASET_NAME = "anvo25/vlms-are-biased"
SPLIT_NAME = "original"

OUTPUT_DIR = Path("/kaggle/working/llava_original_token_original_then_resize")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 2
MIN_CANDIDATE = 0
EXCLUDE_OPTICAL_ILLUSIONS = True

# Use 20 for smoke test; None for full run.
MAX_SAMPLES = None

# Set False to run only numeric/count rows.
RUN_ID_ROWS = True

MAX_NEW_NUMERIC = 3
MAX_NEW_ID = 32

In [4]:
NUMERIC_SUFFIX = "\nAnswer with exactly one integer. No words."

SAVE_EVERY = 5

# Original no-resize safety.
# If original image creates more than this many crops, we skip original generation
# and immediately retry resized.
MAX_CROPS_ORIGINAL = 5

# Resized fallback setting.
MAX_IMAGE_SIDE_RETRY = 448

print("MODEL:", MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUN_ID_ROWS:", RUN_ID_ROWS)
print("MAX_CROPS_ORIGINAL:", MAX_CROPS_ORIGINAL)
print("MAX_IMAGE_SIDE_RETRY:", MAX_IMAGE_SIDE_RETRY)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

MODEL: llava-hf/llava-onevision-qwen2-7b-ov-hf
OUTPUT_DIR: /kaggle/working/llava_original_token_original_then_resize
RUN_ID_ROWS: True
MAX_CROPS_ORIGINAL: 5
MAX_IMAGE_SIDE_RETRY: 448
CUDA: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [5]:
# ============================================================
# 2. Cleanup
# ============================================================

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            try:
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
            except Exception:
                pass
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


cleanup()


In [6]:
# ============================================================
# 3. Load model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

max_memory = None
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    max_memory = {i: "14GiB" for i in range(torch.cuda.device_count())}
    max_memory["cpu"] = "30GiB"

print("Loading model...")
print("max_memory:", max_memory)

model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

print("Loaded model.")
cleanup()

Loading model...
max_memory: {0: '14GiB', 1: '14GiB', 'cpu': '30GiB'}


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/765 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/621 [00:00<?, ?B/s]

Loaded model.


In [7]:
# ============================================================
# 4. Load dataset
# ============================================================

ds = load_dataset(DATASET_NAME, split=SPLIT_NAME)
print("Original rows:", len(ds))

if EXCLUDE_OPTICAL_ILLUSIONS:
    before = len(ds)
    ds = ds.filter(
        lambda x: str(x.get("topic", "")).strip().lower() != "optical illusion"
    )
    print("Removed Optical Illusion rows:", before - len(ds))

if MAX_SAMPLES is not None:
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))
    print("Debug rows:", len(ds))

if "idx" in ds.column_names:
    ds = ds.remove_columns(["idx"])

ds = ds.add_column("idx", list(range(len(ds))))

assert ds["idx"][0] == 0
assert ds["idx"][-1] == len(ds) - 1
assert len(set(ds["idx"])) == len(ds)

print("Final rows:", len(ds))
print("Topics:", Counter(ds["topic"]))

README.md: 0.00B [00:00, ?B/s]

data/main-00000-of-00002.parquet:   0%|          | 0.00/25.0M [00:00<?, ?B/s]

data/main-00001-of-00002.parquet:   0%|          | 0.00/375M [00:00<?, ?B/s]

data/identification-00000-of-00001.parqu(…):   0%|          | 0.00/391M [00:00<?, ?B/s]

data/withtitle-00000-of-00001.parquet:   0%|          | 0.00/451M [00:00<?, ?B/s]

...

Original rows: 458


Filter:   0%|          | 0/458 [00:00<?, ? examples/s]

Removed Optical Illusion rows: 132


Flattening the indices:   0%|          | 0/326 [00:00<?, ? examples/s]

Final rows: 326
Topics: Counter({'Animals': 182, 'Logos': 94, 'Flags': 38, 'Game Boards': 8, 'Chess Pieces': 4})


In [8]:



# ============================================================
# 5. Helpers
# ============================================================

BRACE_RE = re.compile(r"\{([^{}]+)\}")
INT_RE = re.compile(r"-?\d+")


def text(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()


def unbrace(x):
    x = text(x)
    m = BRACE_RE.search(x)
    return m.group(1).strip() if m else x


def first_int(x):
    m = INT_RE.search(unbrace(x))
    return int(m.group(0)) if m else None


def norm(x):
    return unbrace(x).lower().strip().rstrip(".")


def is_numeric(row):
    qtype = text(row.get("type_of_question", "")).lower()
    return ("count" in qtype or "illusion" in qtype) and first_int(row.get("ground_truth")) is not None


def candidate_window(gt):
    gt = int(gt)
    return list(range(max(MIN_CANDIDATE, gt - WINDOW), gt + WINDOW + 1))


def resize_image(image, max_side=MAX_IMAGE_SIDE_RETRY):
    image = image.convert("RGB")
    original_size = image.size

    w, h = image.size
    scale = min(1.0, max_side / max(w, h))

    if scale < 1.0:
        image = image.resize(
            (int(w * scale), int(h * scale)),
            Image.Resampling.LANCZOS,
        )

    return image, original_size, image.size


def chat_prompt(prompt):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ],
    }]
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def entropy(log_probs):
    probs = log_probs.exp()
    return float(-(probs * log_probs).sum().item())


def single_token_meta(value):
    s = str(value)

    for variant in (s, " " + s, "{" + s, "{" + s + "}"):
        ids = tokenizer.encode(variant, add_special_tokens=False)
        if len(ids) == 1:
            return {
                "candidate": int(value),
                "token_id": int(ids[0]),
                "token_variant": variant,
            }

    return None


def is_retryable_error(err):
    err = text(err).lower()
    return (
        "cuda" in err
        or "oom" in err
        or "out of memory" in err
        or "launch failure" in err
        or "too_many_image_crops" in err
    )


def sort_key(row):
    return int(row.get("idx", 10**12))


def read_records(path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        try:
            return pd.read_csv(path).to_dict("records")
        except pd.errors.EmptyDataError:
            return []
    return []


def save_parquet(rows, path):
    df = pd.DataFrame(rows)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str)
    df.to_parquet(path, index=False)


In [9]:
# ============================================================
# 6. Output state
# ============================================================

ALL_CSV = OUTPUT_DIR / "llava_original_model_outputs.csv"
SIGNAL_CSV = OUTPUT_DIR / "llava_original_numeric_signal.csv"
CANDIDATE_CSV = OUTPUT_DIR / "llava_original_numeric_candidates.csv"
SKIP_CSV = OUTPUT_DIR / "llava_original_numeric_signal_skipped.csv"

all_rows = read_records(ALL_CSV)
signal_rows = read_records(SIGNAL_CSV)
candidate_rows = read_records(CANDIDATE_CSV)
skip_rows = read_records(SKIP_CSV)

print("Loaded checkpoint:")
print("all_rows:", len(all_rows))
print("signal_rows:", len(signal_rows))
print("candidate_rows:", len(candidate_rows))
print("skip_rows:", len(skip_rows))


def save_outputs():
    pd.DataFrame(sorted(all_rows, key=sort_key)).to_csv(ALL_CSV, index=False)
    pd.DataFrame(sorted(signal_rows, key=sort_key)).to_csv(SIGNAL_CSV, index=False)

    pd.DataFrame(
        sorted(candidate_rows, key=lambda r: (sort_key(r), int(r.get("candidate", -1))))
    ).to_csv(CANDIDATE_CSV, index=False)

    pd.DataFrame(sorted(skip_rows, key=sort_key)).to_csv(SKIP_CSV, index=False)


Loaded checkpoint:
all_rows: 0
signal_rows: 0
candidate_rows: 0
skip_rows: 0


In [10]:
# ============================================================
# 7. Base row
# ============================================================

def base_row(
    row,
    prompt_used,
    answer,
    error,
    original_size,
    processed_size,
    input_shape=None,
    pixel_shape=None,
    num_crops=None,
    run_mode="original",
    fallback_used=False,
    original_error_before_fallback="",
):
    gt_raw = row.get("ground_truth", "")
    gt_num = first_int(gt_raw)
    pred_num = first_int(answer)

    return {
        "idx": int(row["idx"]),
        "ID": row.get("ID"),
        "image_path": row.get("image_path"),
        "topic": row.get("topic"),
        "sub_topic": row.get("sub_topic"),
        "type_of_question": text(row.get("type_of_question", "")),
        "prompt": text(row.get("prompt", "")),
        "prompt_used_for_generation": prompt_used,
        "ground_truth": gt_raw,
        "ground_truth_num": gt_num,
        "model_answer": answer,
        "pred_norm": norm(answer),
        "gt_norm": norm(gt_raw),
        "pred_num": pred_num,
        "correct_exact": norm(answer) == norm(gt_raw),
        "correct_numeric": (
            int(pred_num) == int(gt_num)
            if pred_num is not None and gt_num is not None
            else False
        ),
        "generation_error": error,

        # Run documentation.
        "run_mode": run_mode,  # "original" or "resized_fallback"
        "fallback_used": bool(fallback_used),
        "original_error_before_fallback": original_error_before_fallback,

        # Image documentation.
        "image_resize_applied": run_mode == "resized_fallback",
        "max_image_side": MAX_IMAGE_SIDE_RETRY if run_mode == "resized_fallback" else None,
        "original_image_width": int(original_size[0]),
        "original_image_height": int(original_size[1]),
        "processed_image_width": int(processed_size[0]),
        "processed_image_height": int(processed_size[1]),

        # Processor diagnostics.
        "input_ids_shape": str(input_shape) if input_shape is not None else "",
        "pixel_values_shape": str(pixel_shape) if pixel_shape is not None else "",
        "num_image_crops": num_crops,
    }

In [11]:
# ============================================================
# 8. Generation attempt
# ============================================================

def generate_attempt(row, need_scores, max_new_tokens, run_mode):
    row = dict(row)

    prompt = text(row.get("prompt", ""))
    prompt_used = prompt + NUMERIC_SUFFIX if is_numeric(row) else prompt

    original_size = (-1, -1)
    processed_size = (-1, -1)
    input_shape = None
    pixel_shape = None
    num_crops = None

    inputs = None
    output = None
    gen_logprobs = None

    try:
        if run_mode == "original":
            image = row["image"].convert("RGB")
            original_size = image.size
            processed_size = image.size
        elif run_mode == "resized_fallback":
            image, original_size, processed_size = resize_image(row["image"], MAX_IMAGE_SIDE_RETRY)
        else:
            raise ValueError(f"Unknown run_mode: {run_mode}")

        inputs = processor(
            images=image,
            text=chat_prompt(prompt_used),
            return_tensors="pt",
        ).to(device)

        input_shape = tuple(inputs["input_ids"].shape)
        pixel_shape = tuple(inputs["pixel_values"].shape) if "pixel_values" in inputs else None
        num_crops = pixel_shape[1] if pixel_shape is not None and len(pixel_shape) == 5 else 1

        print(
            "idx:", row["idx"],
            "mode:", run_mode,
            "orig:", original_size,
            "processed:", processed_size,
            "input_ids:", input_shape,
            "pixel_values:", pixel_shape,
            "crops:", num_crops,
        )

        # Prevent known original no-resize OOM cases.
        if run_mode == "original" and num_crops > MAX_CROPS_ORIGINAL:
            return {
                "row": row,
                "prompt_used": prompt_used,
                "answer": "",
                "gen_ids": None,
                "gen_logprobs": None,
                "error": f"too_many_image_crops_original:{num_crops}",
                "original_size": original_size,
                "processed_size": processed_size,
                "input_shape": input_shape,
                "pixel_shape": pixel_shape,
                "num_crops": num_crops,
                "run_mode": run_mode,
            }

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=need_scores,
                pad_token_id=tokenizer.pad_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        gen_ids = output.sequences[0, prompt_len:].detach().cpu()
        answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        if need_scores and len(output.scores) > 0:
            gen_logprobs = torch.log_softmax(
                torch.stack(output.scores, dim=0).squeeze(1).float(),
                dim=-1,
            ).detach().cpu()

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": answer,
            "gen_ids": gen_ids,
            "gen_logprobs": gen_logprobs,
            "error": "",
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM at idx:", row.get("idx"), "mode:", run_mode)

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": "",
            "gen_ids": None,
            "gen_logprobs": None,
            "error": f"cuda_oom_generation_{run_mode}",
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    except RuntimeError as e:
        err = repr(e)

        if "unspecified launch failure" in err.lower() or "cudaerrorlaunchfailure" in err.lower():
            # CUDA context is likely corrupted. Save and stop.
            err = f"cuda_launch_failure_{run_mode}: " + err
            print("CUDA launch failure at idx:", row.get("idx"))
            print("Saving and stopping. Restart accelerator before continuing.")
            save_outputs()
            raise RuntimeError(err)

        if "cuda" in err.lower() or "oom" in err.lower() or "out of memory" in err.lower():
            err = f"cuda_runtime_generation_{run_mode}: " + err

        print("Runtime error at idx:", row.get("idx"), "mode:", run_mode, err)

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": "",
            "gen_ids": None,
            "gen_logprobs": None,
            "error": err,
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    except Exception as e:
        print("Error at idx:", row.get("idx"), "mode:", run_mode, repr(e))

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": "",
            "gen_ids": None,
            "gen_logprobs": None,
            "error": repr(e),
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    finally:
        try:
            del inputs, output, gen_logprobs
        except Exception:
            pass
        cleanup()


def generate_with_fallback(row, need_scores, max_new_tokens):
    original = generate_attempt(
        row=row,
        need_scores=need_scores,
        max_new_tokens=max_new_tokens,
        run_mode="original",
    )

    if not is_retryable_error(original["error"]):
        original["fallback_used"] = False
        original["original_error_before_fallback"] = ""
        return original

    # Retry with resized image.
    fallback = generate_attempt(
        row=row,
        need_scores=need_scores,
        max_new_tokens=max_new_tokens,
        run_mode="resized_fallback",
    )

    fallback["fallback_used"] = True
    fallback["original_error_before_fallback"] = original["error"]

    return fallback

In [12]:
# ============================================================
# 9. Numeric token analysis
# ============================================================

def analyze_numeric(result):
    row = result["row"]
    gt = first_int(row.get("ground_truth", ""))

    if gt is None:
        return None, [], {"skip_reason": "non_numeric_ground_truth"}

    if result["error"]:
        return None, [], {"skip_reason": "generation_error"}

    if single_token_meta(gt) is None:
        return None, [], {
            "skip_reason": "gt_not_single_token",
            "ground_truth_num": gt,
        }

    candidates = candidate_window(gt)
    valid = []
    skipped = []

    for c in candidates:
        meta = single_token_meta(c)
        if meta is None:
            skipped.append(int(c))
        else:
            valid.append(meta)

    competitors = [c for c in valid if int(c["candidate"]) != int(gt)]

    if not competitors:
        return None, [], {
            "skip_reason": "no_single_token_competitor_in_window",
            "ground_truth_num": gt,
            "candidate_set": json.dumps(candidates),
            "skipped_multitoken_candidates": json.dumps(skipped),
        }

    answer_step = None
    detected_digit = None
    detected_token_text = None

    if result["gen_ids"] is not None:
        for i, token_id in enumerate(result["gen_ids"]):
            tok = tokenizer.decode([int(token_id)], skip_special_tokens=True)
            detected = first_int(tok)
            if detected is not None:
                answer_step = i
                detected_digit = detected
                detected_token_text = tok
                break

    if answer_step is None:
        return None, [], {
            "skip_reason": "no_numeric_answer_step",
            "ground_truth_num": gt,
            "candidate_set": json.dumps(candidates),
            "skipped_multitoken_candidates": json.dumps(skipped),
        }

    if result["gen_logprobs"] is None or answer_step >= result["gen_logprobs"].shape[0]:
        return None, [], {
            "skip_reason": "missing_generation_scores",
            "ground_truth_num": gt,
            "candidate_set": json.dumps(candidates),
            "skipped_multitoken_candidates": json.dumps(skipped),
        }

    lp = result["gen_logprobs"][answer_step].float()
    entropy_token = entropy(lp)

    cand_rows = []
    for meta in valid:
        cand_rows.append({
            **meta,
            "logprob_token": float(lp[meta["token_id"]].item()),
            "is_gt": int(meta["candidate"]) == int(gt),
        })

    gt_rows = [r for r in cand_rows if int(r["candidate"]) == int(gt)]

    if not gt_rows:
        return None, cand_rows, {
            "skip_reason": "gt_missing_from_valid_candidates",
            "ground_truth_num": gt,
            "candidate_set": json.dumps(candidates),
            "skipped_multitoken_candidates": json.dumps(skipped),
        }

    gt_row = gt_rows[0]
    non_gt = [r for r in cand_rows if int(r["candidate"]) != int(gt)]
    proxy = max(non_gt, key=lambda r: r["logprob_token"])

    ranked = sorted(cand_rows, key=lambda r: r["logprob_token"], reverse=True)
    gt_rank = next(i + 1 for i, r in enumerate(ranked) if int(r["candidate"]) == int(gt))

    margin = float(gt_row["logprob_token"] - proxy["logprob_token"])

    base = base_row(
        row=row,
        prompt_used=result["prompt_used"],
        answer=result["answer"],
        error=result["error"],
        original_size=result["original_size"],
        processed_size=result["processed_size"],
        input_shape=result["input_shape"],
        pixel_shape=result["pixel_shape"],
        num_crops=result["num_crops"],
        run_mode=result["run_mode"],
        fallback_used=result.get("fallback_used", False),
        original_error_before_fallback=result.get("original_error_before_fallback", ""),
    )

    signal = {
        **base,
        "detected_digit": detected_digit,
        "detected_token_text": detected_token_text,
        "answer_step": answer_step,
        "candidate_policy": f"GT±{WINDOW}, single-token only",
        "candidate_set": json.dumps(candidates),
        "valid_single_token_candidates": json.dumps([int(c["candidate"]) for c in valid]),
        "skipped_multitoken_candidates": json.dumps(skipped),
        "gt_token_id": gt_row["token_id"],
        "gt_token_variant": gt_row["token_variant"],
        "gt_logprob_token": gt_row["logprob_token"],
        "proxy_bias_value": proxy["candidate"],
        "proxy_bias_token_id": proxy["token_id"],
        "proxy_bias_token_variant": proxy["token_variant"],
        "proxy_bias_logprob_token": proxy["logprob_token"],
        "margin_token": margin,
        "prefers_gt_token": margin > 0,
        "gt_rank_window_token": gt_rank,
        "entropy_token": entropy_token,
        "num_valid_single_token_candidates": len(valid),
        "num_skipped_multitoken_candidates": len(skipped),
        "generated_matches_proxy": (
            detected_digit is not None and int(detected_digit) == int(proxy["candidate"])
        ),
        "full_hallucination": (
            detected_digit is not None
            and int(detected_digit) != int(gt)
            and int(detected_digit) != int(proxy["candidate"])
        ),
    }

    return signal, cand_rows, None

In [13]:


# ============================================================
# 10. Prepare pending rows
# ============================================================

numeric_rows = []
id_rows = []

for row in ds:
    row = dict(row)
    if is_numeric(row):
        numeric_rows.append(row)
    else:
        id_rows.append(row)

done_numeric = {int(r["idx"]) for r in signal_rows if pd.notna(r.get("idx"))}
done_numeric |= {
    int(r["idx"])
    for r in skip_rows
    if r.get("pipeline") == "token" and pd.notna(r.get("idx"))
}

done_outputs = {
    int(r["idx"])
    for r in all_rows
    if pd.notna(r.get("idx"))
}

pending_numeric = [r for r in numeric_rows if int(r["idx"]) not in done_numeric]
pending_id = [r for r in id_rows if int(r["idx"]) not in done_outputs]

if not RUN_ID_ROWS:
    pending_id = []

print("Numeric rows:", len(numeric_rows))
print("ID rows:", len(id_rows))
print("Pending numeric:", len(pending_numeric))
print("Pending ID:", len(pending_id))

Numeric rows: 163
ID rows: 163
Pending numeric: 163
Pending ID: 163


In [14]:

# ============================================================
# 11. Run numeric rows
# ============================================================

processed = len(done_outputs)
next_save = ((processed // SAVE_EVERY) + 1) * SAVE_EVERY

for row in tqdm(pending_numeric, desc="Numeric token rows"):
    result = generate_with_fallback(
        row=row,
        need_scores=True,
        max_new_tokens=MAX_NEW_NUMERIC,
    )

    base = base_row(
        row=row,
        prompt_used=result["prompt_used"],
        answer=result["answer"],
        error=result["error"],
        original_size=result["original_size"],
        processed_size=result["processed_size"],
        input_shape=result["input_shape"],
        pixel_shape=result["pixel_shape"],
        num_crops=result["num_crops"],
        run_mode=result["run_mode"],
        fallback_used=result.get("fallback_used", False),
        original_error_before_fallback=result.get("original_error_before_fallback", ""),
    )

    all_rows.append(base)

    if result["error"]:
        skip_rows.append({
            **base,
            "pipeline": "token",
            "skip_reason": "generation_error",
        })
    else:
        signal, candidates, skip = analyze_numeric(result)

        for c in candidates:
            candidate_rows.append({
                "idx": int(row["idx"]),
                "ID": row.get("ID"),
                "topic": row.get("topic"),
                "sub_topic": row.get("sub_topic"),
                "ground_truth_num": first_int(row.get("ground_truth")),
                "run_mode": result["run_mode"],
                "fallback_used": result.get("fallback_used", False),
                "original_error_before_fallback": result.get("original_error_before_fallback", ""),
                "image_resize_applied": result["run_mode"] == "resized_fallback",
                "max_image_side": MAX_IMAGE_SIDE_RETRY if result["run_mode"] == "resized_fallback" else None,
                "original_image_width": int(result["original_size"][0]),
                "original_image_height": int(result["original_size"][1]),
                "processed_image_width": int(result["processed_size"][0]),
                "processed_image_height": int(result["processed_size"][1]),
                "input_ids_shape": str(result["input_shape"]),
                "pixel_values_shape": str(result["pixel_shape"]),
                "num_image_crops": result["num_crops"],
                **c,
            })

        if signal is not None:
            signal_rows.append(signal)
        else:
            skip_rows.append({
                **base,
                "pipeline": "token",
                **(skip or {}),
            })

    processed += 1

    if processed >= next_save:
        save_outputs()
        print(f"Saved at processed={processed}")
        next_save += SAVE_EVERY

    cleanup()

Numeric token rows:   0%|          | 0/163 [00:00<?, ?it/s]

idx: 0 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 2 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 4 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 6 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 8 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
Saved at processed=5
idx: 10 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 12 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 14 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5

In [15]:
# ============================================================
# 12. Run ID rows
# ============================================================

if RUN_ID_ROWS:
    for row in tqdm(pending_id, desc="ID rows"):
        result = generate_with_fallback(
            row=row,
            need_scores=False,
            max_new_tokens=MAX_NEW_ID,
        )

        base = base_row(
            row=row,
            prompt_used=result["prompt_used"],
            answer=result["answer"],
            error=result["error"],
            original_size=result["original_size"],
            processed_size=result["processed_size"],
            input_shape=result["input_shape"],
            pixel_shape=result["pixel_shape"],
            num_crops=result["num_crops"],
            run_mode=result["run_mode"],
            fallback_used=result.get("fallback_used", False),
            original_error_before_fallback=result.get("original_error_before_fallback", ""),
        )

        all_rows.append(base)

        if result["error"]:
            skip_rows.append({
                **base,
                "pipeline": "generation",
                "skip_reason": "generation_error",
            })

        processed += 1

        if processed >= next_save:
            save_outputs()
            print(f"Saved at processed={processed}")
            next_save += SAVE_EVERY

        cleanup()

ID rows:   0%|          | 0/163 [00:00<?, ?it/s]

idx: 1 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2736) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 3 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2736) pixel_values: (1, 5, 3, 384, 384) crops: 5
Saved at processed=165
idx: 5 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2736) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 7 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2736) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 9 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2736) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 11 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2736) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 13 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2736) pixel_values: (1, 5, 3, 384, 384) crops: 5
Saved at processed=170
idx: 15 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 

In [16]:

# ============================================================
# 13. Finalize
# ============================================================

if all_rows:
    all_rows = (
        pd.DataFrame(all_rows)
        .drop_duplicates("idx", keep="last")
        .sort_values("idx")
        .to_dict("records")
    )

if signal_rows:
    signal_rows = (
        pd.DataFrame(signal_rows)
        .drop_duplicates("idx", keep="last")
        .sort_values("idx")
        .to_dict("records")
    )

if candidate_rows:
    candidate_rows = (
        pd.DataFrame(candidate_rows)
        .drop_duplicates(["idx", "candidate"], keep="last")
        .sort_values(["idx", "candidate"])
        .to_dict("records")
    )

if skip_rows:
    skip_df = pd.DataFrame(skip_rows)
    dedup_cols = ["idx", "pipeline"] if "pipeline" in skip_df.columns else ["idx"]
    skip_rows = (
        skip_df
        .drop_duplicates(dedup_cols, keep="last")
        .sort_values("idx")
        .to_dict("records")
    )

# Remove skipped rows where a signal exists.
if signal_rows and skip_rows:
    signal_idx = {int(r["idx"]) for r in signal_rows}
    skip_rows = [
        r for r in skip_rows
        if int(r.get("idx", -1)) not in signal_idx
    ]

save_outputs()

save_parquet(all_rows, OUTPUT_DIR / "llava_original_model_outputs.parquet")
save_parquet(signal_rows, OUTPUT_DIR / "llava_original_numeric_signal.parquet")
save_parquet(candidate_rows, OUTPUT_DIR / "llava_original_numeric_candidates.parquet")
save_parquet(skip_rows, OUTPUT_DIR / "llava_original_numeric_signal_skipped.parquet")

ZIP_BASE = "/kaggle/working/llava_original_token_original_then_resize"
ZIP_PATH = ZIP_BASE + ".zip"

if Path(ZIP_PATH).exists():
    Path(ZIP_PATH).unlink()

shutil.make_archive(ZIP_BASE, "zip", OUTPUT_DIR)

'/kaggle/working/llava_original_token_original_then_resize.zip'

In [17]:
# ============================================================
# 14. Summary
# ============================================================

all_df = pd.DataFrame(all_rows)
sig_df = pd.DataFrame(signal_rows)
skip_df = pd.DataFrame(skip_rows)
cand_df = pd.DataFrame(candidate_rows)

print("\nDone.")
print("All output rows:", len(all_df))
print("Signal rows:", len(sig_df))
print("Candidate rows:", len(cand_df))
print("Skipped rows:", len(skip_df))
print("ZIP:", ZIP_PATH)

if len(all_df):
    print("\nGeneration errors:")
    print(
        all_df["generation_error"]
        .fillna("")
        .replace("", "NO_ERROR")
        .value_counts(dropna=False)
        .head(30)
    )

    print("\nRun modes:")
    print(all_df["run_mode"].value_counts(dropna=False))

    print("\nFallback used:")
    print(all_df["fallback_used"].value_counts(dropna=False))

if len(sig_df):
    print("\nSignal rows:", len(sig_df))
    print("Prefers GT rate:", sig_df["prefers_gt_token"].mean())
    print(sig_df["margin_token"].describe())

    print("\nSignal rows by run mode:")
    print(sig_df["run_mode"].value_counts(dropna=False))

if len(skip_df):
    print("\nSkip reasons:")
    print(
        skip_df["skip_reason"]
        .fillna("")
        .replace("", "NO_REASON")
        .value_counts(dropna=False)
        .head(30)
    )

print("\nSaved files:")
print(ALL_CSV)
print(SIGNAL_CSV)
print(CANDIDATE_CSV)
print(SKIP_CSV)
print(ZIP_PATH)


Done.
All output rows: 326
Signal rows: 154
Candidate rows: 761
Skipped rows: 9
ZIP: /kaggle/working/llava_original_token_original_then_resize.zip

Generation errors:
generation_error
NO_ERROR    326
Name: count, dtype: int64

Run modes:
run_mode
resized_fallback    288
original             38
Name: count, dtype: int64

Fallback used:
fallback_used
True     288
False     38
Name: count, dtype: int64

Signal rows: 154
Prefers GT rate: 0.8376623376623377
count    154.000000
mean       4.284852
std        3.054701
min       -4.921875
25%        1.175781
50%        5.828125
75%        6.546875
max        7.671875
Name: margin_token, dtype: float64

Signal rows by run mode:
run_mode
resized_fallback    140
original             14
Name: count, dtype: int64

Skip reasons:
skip_reason
gt_not_single_token    9
Name: count, dtype: int64

Saved files:
/kaggle/working/llava_original_token_original_then_resize/llava_original_model_outputs.csv
/kaggle/working/llava_original_token_original_then_resi